<a href="https://colab.research.google.com/github/Mambwe-LC/db-unza26-csc4792-Mufulira-Municipal-Council-group-8/blob/main/db_unza26_csc4792_Mufulira_Municipal_Council.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 4792 — Data Mining and Warehousing
## Mufulira Municipal Council Dataset — Group 8

**Source:** [Mufulira Municipal Council](https://www.mufuliracouncil.gov.zm)
**Repo:** [db-unza26-csc4792-Mufulira-Municipal-Council-group-8](https://github.com/Mambwe-LC/db-unza26-csc4792-Mufulira-Municipal-Council-group-8)

This notebook is a **lightweight orchestrator**, not a copy of the pipeline. All scraping and
cleaning logic lives in the repo's `scrapping/` and `cleaning/` scripts (tested, version
controlled, and independently runnable via `python scrap.py` / `python clean.py`). This
notebook clones the repo, then either:

- **Loads the pre-built raw and clean datasets already committed to the repo** (fast, the
  default), or
- **Re-runs the actual extraction and cleaning scripts from scratch** (slow — real network
  scraping + OCR — only if you set `REGENERATE_DATASETS = True` below), for anyone who wants
  to see exactly how the datasets were produced.

Either way, the notebook then documents the **raw → clean transformation** (what changed and
why) and runs EDA on the final cleaned datasets.


## 1. Environment Setup

In [ ]:
import os

# ============================================================
# CONFIG
# ============================================================
# False (default): just load the datasets already committed to the repo.
# True: actually re-run the scraping + cleaning pipeline from scratch
#       (needs internet access and takes several minutes - OCR is slow).
REGENERATE_DATASETS = False

REPO_URL = "https://github.com/Mambwe-LC/db-unza26-csc4792-Mufulira-Municipal-Council-group-8"
REPO_NAME = "db-unza26-csc4792-Mufulira-Municipal-Council-group-8"

# ============================================================
# MOUNT GOOGLE DRIVE (optional, but recommended so re-generated
# datasets persist across Colab sessions instead of disappearing
# when the runtime resets)
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

WORKSPACE = "/content/drive/MyDrive/CSC4792-Group8-Mufulira-Council"
os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)

# ============================================================
# CLONE THE REPO
# ============================================================
# Re-clone fresh each run so the notebook always reflects exactly
# what's committed - not a stale local copy from a previous session.

import shutil
if os.path.exists(REPO_NAME):
    shutil.rmtree(REPO_NAME)

!git clone {REPO_URL}.git

os.chdir(REPO_NAME)
print("\nWorking directory:", os.getcwd())

# ============================================================
# PYTHON DEPENDENCIES
# ============================================================
# Always needed, even in the fast "just load the data" path.

!pip -q install pandas matplotlib

if REGENERATE_DATASETS:
    # Only needed for the actual extraction step (OCR + PDF table parsing).
    !apt-get -qq update
    !apt-get -qq install -y tesseract-ocr poppler-utils
    !pip -q install requests beautifulsoup4 pdfplumber openpyxl urllib3 pytesseract pdf2image

    import subprocess
    print(subprocess.run(["tesseract", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
    print(subprocess.run(["pdftoppm", "-v"], capture_output=True, text=True).stderr.splitlines()[0])

print("\nSetup complete. REGENERATE_DATASETS =", REGENERATE_DATASETS)


## 2. Data Sources

Four groups of source documents, all published on `https://www.mufuliracouncil.gov.zm`:

| Source | Format | What it contains |
|---|---|---|
| **Integrated Development Plan (IDP)** | Digital PDF, table-based | Ward/constituency demographics, health facilities, public consultation issues, capital investment framework |
| **CDF Tracker — Community Projects** | Scanned PDFs (image-based), one per constituency | Project number, ward, sector, scope of work, engineer's estimate and approved amount |
| **CDF Tracker — Skills Development Applicants** | Mix of digital and scanned PDFs (2025–2026) | Applicant name, NRC, gender, date of birth, study programme/institution, guardian |
| **Approved Budgets** (2023–2026) | Digital PDFs, table-based | Revenue by source, payments by category, budget vs. actual figures |

Extraction uses a **hybrid strategy**: `pdfplumber` for digital PDFs with real text/table
structure, and OCR (`pytesseract` + `pdf2image`) as a fallback for scanned pages.


## 3. Extraction & Cleaning — Reference Code (optional)

**This section only runs if `REGENERATE_DATASETS = True`.** It calls the repo's actual
scraping scripts (`scrapping/`) and cleaning scripts (`cleaning/`) directly — nothing here is
a copy of their logic, it's the real code, imported and executed. This is included purely for
transparency and marking: it shows exactly how `data/raw/` and `data/clean/` were produced.
**You don't need to run this to explore the data** — skip straight to Section 4.

Each extraction script writes its raw output using a relative path, so after running it we
move any freshly-created `db-unza26-csc4792*.csv` file sitting in the repo root into
`data/raw/`, in case the script itself hasn't been updated to write there directly.

In [ ]:
import sys, glob, shutil, os

if REGENERATE_DATASETS:
    os.makedirs(os.path.join("data", "raw"), exist_ok=True)
    os.makedirs(os.path.join("data", "clean"), exist_ok=True)

    for sub in ["scrapping/idp_scraping", "scrapping/cdf_dataset_scrapping",
                "scrapping/budget_scraper", "cleaning"]:
        full = os.path.join(os.getcwd(), sub)
        if full not in sys.path:
            sys.path.append(full)

    def collect_new_raw_outputs():
        """Move any freshly-written db-unza26-csc4792*.csv sitting in the
        repo root into data/raw/, in case a script's OUTPUT_DIR still
        points at '.' instead of 'data/raw'."""
        for f in glob.glob("db-unza26-csc4792*mufulira*.csv"):
            dest = os.path.join("data", "raw", f)
            shutil.move(f, dest)
            print(f"  moved {f} -> {dest}")
else:
    print("REGENERATE_DATASETS is False - skipping (see Section 4 to load pre-built data).")


### 3.1 IDP — Governance Datasets

In [ ]:
if REGENERATE_DATASETS:
    from idp_scraper import main as extract_idp
    extract_idp()
    collect_new_raw_outputs()


### 3.2 CDF Community Projects

In [ ]:
if REGENERATE_DATASETS:
    from cdf_comm_projects_scraper import main as extract_cdf_projects
    extract_cdf_projects()
    collect_new_raw_outputs()


### 3.3 CDF Skills Development Applicants

In [ ]:
if REGENERATE_DATASETS:
    from cdf_skill_dev_applicants_scraper import main as extract_cdf_skills
    extract_cdf_skills()
    collect_new_raw_outputs()


### 3.4 Approved Budgets & Revenue

In [ ]:
if REGENERATE_DATASETS:
    from budget_scraper import main as extract_budget
    extract_budget()
    collect_new_raw_outputs()


### 3.5 Data Cleaning & Preprocessing

Runs the repo's `cleaning/` scripts in the required order (wards demographics must run before
health facilities — the latter backfills `Constituency` from the former's cleaned output).

In [ ]:
if REGENERATE_DATASETS:
    from clean_administrative_wards_demographics import main as clean_wards
    from clean_cdf_projects import main as clean_cdf_projects_fn
    from clean_cdf_skills_applicants import main as clean_cdf_skills_fn
    from clean_health_facilities import main as clean_health_fn
    from clean_master_capital_investment_framework import main as clean_capital_fn
    from clean_ward_public_consultation_issues import main as clean_consult_fn
    from clean_budget_revenue import main as clean_budget_rev_fn
    from clean_budget_raw_tables import main as clean_budget_raw_fn

    clean_wards()
    clean_cdf_projects_fn()
    clean_cdf_skills_fn()
    clean_health_fn()
    clean_capital_fn()
    clean_consult_fn()
    clean_budget_rev_fn()
    clean_budget_raw_fn()


## 4. Load Datasets

Loads both the raw and clean CSVs from `data/raw/` and `data/clean/` — either the versions
just regenerated above, or (the default) the ones already committed to the repo.

In [ ]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# key -> (raw filename, clean filename)
# NOTE: the two budget files kept an underscore ("csc4792_mufulira") in the
# raw extraction; the clean pipeline corrects this to match the assignment's
# required naming convention ("csc4792-mufulira").
DATASET_FILES = {
    "administrative_wards_demographics": (
        "db-unza26-csc4792-mufulira_administrative_wards_demographics.csv",
        "db-unza26-csc4792-mufulira_administrative_wards_demographics.csv"),
    "cdf_projects": (
        "db-unza26-csc4792-mufulira_cdf_projects.csv",
        "db-unza26-csc4792-mufulira_cdf_projects.csv"),
    "cdf_skills_applicants": (
        "db-unza26-csc4792-mufulira_cdf_skills_applicants.csv",
        "db-unza26-csc4792-mufulira_cdf_skills_applicants.csv"),
    "health_facilities": (
        "db-unza26-csc4792-mufulira_health_facilities.csv",
        "db-unza26-csc4792-mufulira_health_facilities.csv"),
    "master_capital_investment_framework": (
        "db-unza26-csc4792-mufulira_master_capital_investment_framework.csv",
        "db-unza26-csc4792-mufulira_master_capital_investment_framework.csv"),
    "ward_public_consultation_issues": (
        "db-unza26-csc4792-mufulira_ward_public_consultation_issues.csv",
        "db-unza26-csc4792-mufulira_ward_public_consultation_issues.csv"),
    "budget_revenue_2023_2026": (
        "db-unza26-csc4792_mufulira_budget_revenue_2023_2026.csv",
        "db-unza26-csc4792-mufulira_budget_revenue_2023_2026.csv"),
    "budget_raw_tables_2023_2026": (
        "db-unza26-csc4792_mufulira_budget_raw_tables_2023_2026.csv",
        "db-unza26-csc4792-mufulira_budget_raw_tables_2023_2026.csv"),
}

raw_datasets = {}
datasets = {}  # the final, cleaned versions

for key, (raw_name, clean_name) in DATASET_FILES.items():
    raw_path = os.path.join("data", "raw", raw_name)
    clean_path = os.path.join("data", "clean", clean_name)

    if os.path.exists(raw_path):
        raw_datasets[key] = pd.read_csv(raw_path, sep="|", dtype=str, encoding="utf-8-sig")
    else:
        print(f"[WARN] raw file missing: {raw_path}")

    if os.path.exists(clean_path):
        datasets[key] = pd.read_csv(clean_path, sep="|", dtype=str, encoding="utf-8-sig")
    else:
        print(f"[WARN] clean file missing: {clean_path}")

print(f"Loaded {len(raw_datasets)} raw and {len(datasets)} clean dataset(s).")


## 5. Data Cleaning & Preprocessing — Raw vs. Clean Comparison

This is the evidence that the datasets were actually cleaned and preprocessed, not just
scraped: for each dataset, row count and missing-value rate before vs. after, plus how key
categorical columns got standardized (e.g. inconsistent Gender entries collapsing into `M`/`F`).
Uses the same missing-value sentinel list (`Unspecified`, `N/A`, ...) the cleaning scripts
themselves used, via `cleaning_utils`, so the "before" side is judged by the same definition
of "missing" as the "after" side.

In [ ]:
import sys, os

sys.path.append(os.path.join(os.getcwd(), "cleaning"))
from cleaning_utils import normalize_missing_sentinels

def compare_raw_clean(name):
    raw = raw_datasets.get(name)
    clean = datasets.get(name)
    if raw is None or clean is None:
        print(f"Skipping {name}: missing raw or clean version.")
        return

    raw_norm = normalize_missing_sentinels(raw)  # same sentinel handling the cleaning scripts used

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)
    print(f"Rows:    raw={raw.shape[0]:>6}   clean={clean.shape[0]:>6}   "
          f"(net change: {clean.shape[0] - raw.shape[0]:+d})")
    print(f"Columns: raw={raw.shape[1]:>6}   clean={clean.shape[1]:>6}")

    common_cols = [c for c in clean.columns if c in raw_norm.columns]
    report = pd.DataFrame([
        {
            "column": c,
            "raw_%_missing": round(raw_norm[c].isna().mean() * 100, 1),
            "clean_%_missing": round(clean[c].isna().mean() * 100, 1),
        }
        for c in common_cols
    ])
    display(report)

    # Show standardization on a couple of common categorical columns, where present
    for c in ["Gender", "Constituency", "Sector"]:
        if c in common_cols:
            raw_vals = sorted(raw_norm[c].dropna().unique().tolist())
            clean_vals = sorted(clean[c].dropna().unique().tolist())
            print(f"\nUnique values in '{c}':")
            print(f"  raw   ({len(raw_vals)}): {raw_vals[:15]}")
            print(f"  clean ({len(clean_vals)}): {clean_vals[:15]}")

    new_cols = [c for c in clean.columns if c not in raw.columns]
    if new_cols:
        print(f"\nColumns added during preprocessing: {new_cols}")
    dropped_cols = [c for c in raw.columns if c not in clean.columns]
    if dropped_cols:
        print(f"Columns dropped during preprocessing: {dropped_cols}")


for name in datasets:
    compare_raw_clean(name)


## 6. Exploratory Data Analysis (EDA)

A first look at each cleaned dataset — distributions, category breakdowns, and simple totals.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 5)

def to_numeric(series):
    """Strip currency symbols/commas/whitespace and coerce to numeric."""
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True),
        errors="coerce"
    )

def bar_from_counts(counts, title, xlabel, ylabel="Count", rot=45):
    ax = counts.plot(kind="bar")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.xticks(rotation=rot, ha="right")
    plt.tight_layout()
    plt.show()


### 6.1 CDF Community Projects

In [ ]:
df = datasets.get("cdf_projects")

if df is not None and not df.empty:
    print(f"{len(df)} CDF community project records across "
          f"{df['Constituency'].nunique()} constituencies.\n")

    constituency_counts = df["Constituency"].dropna().value_counts()
    if not constituency_counts.empty:
        bar_from_counts(constituency_counts,
                         "CDF Community Projects by Constituency", "Constituency")

    if "Sector" in df.columns:
        sector_counts = df["Sector"].dropna().value_counts()
        if not sector_counts.empty:
            bar_from_counts(sector_counts, "CDF Community Projects by Sector", "Sector")

    if "Approved_Amount_ZMW" in df.columns:
        amounts = to_numeric(df["Approved_Amount_ZMW"]).dropna()
        if not amounts.empty:
            print(amounts.describe().to_string())
            amounts.plot(kind="hist", bins=20, title="Approved Amount (ZMW) Distribution")
            plt.xlabel("Approved Amount (ZMW)")
            plt.tight_layout()
            plt.show()

    if "Ward" in df.columns:
        ward_counts = df["Ward"].dropna().value_counts().head(10)
        if not ward_counts.empty:
            bar_from_counts(ward_counts, "Top 10 Wards by Project Count", "Ward")
else:
    print("No CDF community projects data available.")


### 6.2 CDF Skills Development Applicants

In [ ]:
df = datasets.get("cdf_skills_applicants")

if df is not None and not df.empty:
    print(f"{len(df)} skills development applicant records.\n")

    constituency_counts = df["Constituency"].dropna().value_counts()
    if not constituency_counts.empty:
        bar_from_counts(constituency_counts,
                         "Skills Development Applicants by Constituency", "Constituency")

    if "Gender" in df.columns:
        gender_counts = df["Gender"].dropna().value_counts()
        if not gender_counts.empty:
            bar_from_counts(gender_counts, "Applicants by Gender", "Gender", rot=0)
else:
    print("No skills applicants data available.")


### 6.3 Approved Budgets & Revenue (2023–2026)

In [ ]:
df = datasets.get("budget_revenue_2023_2026")

if df is not None and not df.empty:
    df = df.copy()
    df["budget_amount_num"] = to_numeric(df["budget_amount"])

    print(f"{len(df)} budget line items across {df['year'].nunique()} year(s).\n")

    by_year = df.groupby("year")["budget_amount_num"].sum().sort_index()
    if not by_year.empty:
        bar_from_counts(by_year, "Total Budget Amount (ZMW) by Year", "Year",
                         ylabel="Total Amount (ZMW)", rot=0)

    if "data_type" in df.columns:
        by_type = df.groupby("data_type")["budget_amount_num"].sum().sort_values(ascending=False)
        if not by_type.empty:
            bar_from_counts(by_type, "Total Budget Amount by Data Type", "Data Type",
                             ylabel="Total Amount (ZMW)", rot=0)

    if "budget_status" in df.columns:
        status_counts = df["budget_status"].dropna().value_counts()
        if not status_counts.empty:
            bar_from_counts(status_counts, "Line Items by Budget Status", "Status", rot=0)
else:
    print("No budget/revenue data available.")


### 6.4 IDP — Wards & Demographics

In [ ]:
df = datasets.get("administrative_wards_demographics")

if df is not None and not df.empty:
    df = df.copy()
    for col in ["Total_Population", "Male_Population", "Female_Population"]:
        if col in df.columns:
            df[col] = to_numeric(df[col])

    print(f"{len(df)} ward-level records.\n")

    if {"Total_Population", "Ward_Name", "Level"}.issubset(df.columns):
        ward_rows = df[df["Level"] == "Ward"].dropna(subset=["Total_Population"])
        pop_by_ward = ward_rows.set_index("Ward_Name")["Total_Population"].sort_values(ascending=False)
        if not pop_by_ward.empty:
            bar_from_counts(pop_by_ward, "Total Population by Ward", "Ward", ylabel="Population")
else:
    print("No wards/demographics data available.")


### 6.5 IDP — Health Facilities

In [ ]:
df = datasets.get("health_facilities")

if df is not None and not df.empty:
    print(f"{len(df)} health facility records.\n")

    if "Facility_Type" in df.columns:
        type_counts = df["Facility_Type"].dropna().value_counts()
        if not type_counts.empty:
            bar_from_counts(type_counts, "Health Facilities by Type", "Facility Type")

    if "Operational_Status" in df.columns:
        status_counts = df["Operational_Status"].dropna().value_counts()
        if not status_counts.empty:
            bar_from_counts(status_counts, "Health Facilities by Operational Status", "Status", rot=0)
else:
    print("No health facilities data available.")


### 6.6 IDP — Master Capital Investment Framework

In [ ]:
df = datasets.get("master_capital_investment_framework")

if df is not None and not df.empty:
    df = df.copy()
    if "Cost_ZMW" in df.columns:
        df["Cost_ZMW_num"] = to_numeric(df["Cost_ZMW"])

    print(f"{len(df)} planned capital investment line items.\n")

    if "Sector" in df.columns:
        sector_counts = df["Sector"].dropna().value_counts()
        if not sector_counts.empty:
            bar_from_counts(sector_counts, "Planned Projects by Sector", "Sector")

    if "Cost_ZMW_num" in df.columns and "Sector" in df.columns:
        cost_by_sector = df.groupby("Sector")["Cost_ZMW_num"].sum().sort_values(ascending=False).head(10)
        if not cost_by_sector.empty:
            bar_from_counts(cost_by_sector, "Top 10 Sectors by Total Planned Cost (ZMW)", "Sector",
                             ylabel="Total Cost (ZMW)")
else:
    print("No capital investment framework data available.")


### 6.7 IDP — Ward Public Consultation Issues

In [ ]:
df = datasets.get("ward_public_consultation_issues")

if df is not None and not df.empty:
    print(f"{len(df)} public consultation issue records.\n")
    if "Sector" in df.columns:
        sector_counts = df["Sector"].dropna().value_counts()
        if not sector_counts.empty:
            bar_from_counts(sector_counts, "Consultation Issues by Sector", "Sector")
        else:
            print("'Sector' has no non-missing values in this dataset - skipping chart.")
else:
    print("No ward public consultation data available.")


## 7. Data Dictionary

| File | Source | Description |
|---|---|---|
| `db-unza26-csc4792-mufulira_administrative_wards_demographics.csv` | IDP | Ward/constituency demographics, with derived `Level` (District/Constituency/Ward) |
| `db-unza26-csc4792-mufulira_health_facilities.csv` | IDP | Health facility records |
| `db-unza26-csc4792-mufulira_ward_public_consultation_issues.csv` | IDP | Issues raised during ward public consultations |
| `db-unza26-csc4792-mufulira_master_capital_investment_framework.csv` | IDP | Capital investment framework line items |
| `db-unza26-csc4792-mufulira_cdf_projects.csv` | CDF Tracker | CDF community projects by constituency, ward, sector, and approved amount |
| `db-unza26-csc4792-mufulira_cdf_skills_applicants.csv` | CDF Tracker | Skills development bursary applicants by constituency |
| `db-unza26-csc4792-mufulira_budget_revenue_2023_2026.csv` | Approved Budgets | Cleaned revenue & spending, 2023–2026 |
| `db-unza26-csc4792-mufulira_budget_raw_tables_2023_2026.csv` | Approved Budgets | Raw extracted budget tables (traceability) |

**Next steps (outside this notebook):**
1. Upload the `data/clean/` CSVs to your Kaggle dataset.
2. Confirm `lighton.phiri@gmail.com` has been added as a GitHub collaborator.
3. Use Section 5 (raw vs. clean comparison) and Section 6 (EDA) as source material for the
   Data Description Paper's methodology and results sections.
